In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..")))

import pandas as pd
from src.preprocessing import (
    format_time_series,
    reindex_time_series,
    impute_time_series,
    calculate_hour_dayofweek_month,
    calculate_holiday_weekend,
    calculate_lags,
    calculate_rolling_mean_std,
    calculate_fourier_terms,
    split_data,
    model_features,
    encode_cyclical,
    calculate_weather_features,
)

In [2]:
pjme = (
    pd.read_csv("../data/PJME_hourly.csv")
    .pipe(format_time_series)
    .pipe(reindex_time_series)
    .pipe(impute_time_series)
)

== Date Range ==
Start: 2002-01-01 01:00:00
End:   2018-08-03 00:00:00
Expected hourly observations: 145,392
== Duplicate Timestamps: 4 ==
                      energy
Datetime                    
2014-11-02 02:00:00  23755.0
2015-11-01 02:00:00  21171.0
2016-11-06 02:00:00  21692.0
2017-11-05 02:00:00  20666.0
Rows after reindexing: 145,392


In [3]:
print(pjme.info())
print(pjme.describe())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 145392 entries, 2002-01-01 01:00:00 to 2018-08-03 00:00:00
Freq: h
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   energy  145392 non-null  float64
dtypes: float64(1)
memory usage: 2.2 MB
None
              energy
count  145392.000000
mean    32078.925072
std      6464.286922
min     14544.000000
25%     27571.000000
50%     31420.000000
75%     35647.000000
max     62009.000000


In [4]:
pjme = calculate_hour_dayofweek_month(pjme)
print(pjme.head(10))
print(f"hour: {pjme['hour'].unique()}")
print(f"dayofweek: {pjme['dayofweek'].unique()}")
print(f"month: {pjme['month'].unique()}")

                      energy  hour  dayofweek  month
2002-01-01 01:00:00  30393.0     1          1      1
2002-01-01 02:00:00  29265.0     2          1      1
2002-01-01 03:00:00  28357.0     3          1      1
2002-01-01 04:00:00  27899.0     4          1      1
2002-01-01 05:00:00  28057.0     5          1      1
2002-01-01 06:00:00  28654.0     6          1      1
2002-01-01 07:00:00  29308.0     7          1      1
2002-01-01 08:00:00  29595.0     8          1      1
2002-01-01 09:00:00  29943.0     9          1      1
2002-01-01 10:00:00  30692.0    10          1      1
hour: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23  0]
dayofweek: [1 2 3 4 5 6 0]
month: [ 1  2  3  4  5  6  7  8  9 10 11 12]


In [5]:
pjme = calculate_holiday_weekend(pjme)
pjme[["is_holiday", "is_weekend"]].info()
print(pjme["is_holiday"].value_counts())
print(pjme["is_weekend"].value_counts())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 145392 entries, 2002-01-01 01:00:00 to 2018-08-03 00:00:00
Freq: h
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   is_holiday  145392 non-null  int8 
 1   is_weekend  145392 non-null  int8 
dtypes: int8(2)
memory usage: 1.4 MB
is_holiday
0    141001
1      4391
Name: count, dtype: int64
is_weekend
0    103872
1     41520
Name: count, dtype: int64


In [6]:
pjme = encode_cyclical(pjme)
print(pjme.columns)
pjme[["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]].head(3)

Index(['energy', 'hour', 'dayofweek', 'month', 'is_holiday', 'is_weekend',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos'],
      dtype='object')


,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
2002-01-01 01:00:00,0.258819,0.965926,0.781831,0.62349,0.5,0.866025
2002-01-01 02:00:00,0.500000,0.866025,0.781831,0.62349,0.5,0.866025
2002-01-01 03:00:00,0.707107,0.707107,0.781831,0.62349,0.5,0.866025


In [7]:
pjme = calculate_fourier_terms(pjme, periods=[24, 168, 8766], ks=[3, 3, 2])
print(pjme.columns)
pjme[
    ["sin_24_k3", "cos_24_k3", "sin_168_k3", "cos_168_k3", "sin_8766_k2", "cos_8766_k2"]
].head(3)

Index(['energy', 'hour', 'dayofweek', 'month', 'is_holiday', 'is_weekend',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
       'sin_24_k3', 'cos_24_k3', 'sin_168_k3', 'cos_168_k3', 'sin_8766_k2',
       'cos_8766_k2'],
      dtype='object')


,sin_24_k3,cos_24_k3,sin_168_k3,cos_168_k3,sin_8766_k2,cos_8766_k2
2002-01-01 01:00:00,0.000000,1.000000e+00,0.000000,1.000000,0.000000,1.000000
2002-01-01 02:00:00,0.707107,7.071068e-01,0.111964,0.993712,0.001434,0.999999
2002-01-01 03:00:00,1.000000,6.123234e-17,0.222521,0.974928,0.002867,0.999996


In [8]:
pjme = calculate_lags(pjme, lags=[1, 2, 24, 48, 168, 336])
print(pjme.columns)
pjme[["lag_1", "lag_2", "lag_24", "lag_48", "lag_168", "lag_336"]].dropna().head(3)

Index(['energy', 'hour', 'dayofweek', 'month', 'is_holiday', 'is_weekend',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
       'sin_24_k3', 'cos_24_k3', 'sin_168_k3', 'cos_168_k3', 'sin_8766_k2',
       'cos_8766_k2', 'lag_1', 'lag_2', 'lag_24', 'lag_48', 'lag_168',
       'lag_336'],
      dtype='object')


,lag_1,lag_2,lag_24,lag_48,lag_168,lag_336
2002-01-15 01:00:00,27483.0,30005.0,26059.0,25052.0,29445.0,30393.0
2002-01-15 02:00:00,25643.0,27483.0,25487.0,24007.0,28670.0,29265.0
2002-01-15 03:00:00,24967.0,25643.0,25363.0,23441.0,28375.0,28357.0


In [9]:
pjme = calculate_rolling_mean_std(pjme, periods=[24, 168])
print(pjme.columns)
pjme[["roll_mean_24", "roll_std_24", "roll_mean_168", "roll_std_168"]].dropna().head(3)

Index(['energy', 'hour', 'dayofweek', 'month', 'is_holiday', 'is_weekend',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
       'sin_24_k3', 'cos_24_k3', 'sin_168_k3', 'cos_168_k3', 'sin_8766_k2',
       'cos_8766_k2', 'lag_1', 'lag_2', 'lag_24', 'lag_48', 'lag_168',
       'lag_336', 'roll_mean_24', 'roll_std_24', 'roll_mean_168',
       'roll_std_168'],
      dtype='object')


,roll_mean_24,roll_std_24,roll_mean_168,roll_std_168
2002-01-08 00:00:00,33452.583333,4559.767709,32519.511905,3857.950565
2002-01-08 01:00:00,33560.208333,4425.965952,32513.869048,3861.770954
2002-01-08 02:00:00,33672.458333,4256.159403,32510.327381,3865.039821


In [10]:
pjme = calculate_weather_features(pjme)
print(pjme["mean_cdh"].value_counts(normalize=True))
print(pjme["mean_hdh"].value_counts(normalize=True))

mean_cdh
0.0      0.548586
0.06     0.012153
0.18     0.007029
0.28     0.004127
0.24     0.003728
           ...   
15.42    0.000007
19.22    0.000007
18.66    0.000007
17.1     0.000007
11.28    0.000007
Name: proportion, Length: 2034, dtype: Float64
mean_hdh
0.0      0.344648
0.04     0.011871
0.16     0.007456
0.26     0.004175
0.08     0.003006
           ...   
17.84    0.000007
3.12     0.000007
32.9     0.000007
33.02    0.000007
4.36     0.000007
Name: proportion, Length: 2526, dtype: Float64


In [11]:
pjme.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 145392 entries, 2002-01-01 01:00:00 to 2018-08-03 00:00:00
Freq: h
Data columns (total 40 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   energy         145392 non-null  float64
 1   hour           145392 non-null  int32  
 2   dayofweek      145392 non-null  int32  
 3   month          145392 non-null  int32  
 4   is_holiday     145392 non-null  int8   
 5   is_weekend     145392 non-null  int8   
 6   hour_sin       145392 non-null  float64
 7   hour_cos       145392 non-null  float64
 8   dow_sin        145392 non-null  float64
 9   dow_cos        145392 non-null  float64
 10  month_sin      145392 non-null  float64
 11  month_cos      145392 non-null  float64
 12  sin_24_k3      145392 non-null  float64
 13  cos_24_k3      145392 non-null  float64
 14  sin_168_k3     145392 non-null  float64
 15  cos_168_k3     145392 non-null  float64
 16  sin_8766_k2    145392 non-null  

In [12]:
train, val, test = split_data(pjme)

print(f"train shape: {train.shape}")
print(f"val shape: {val.shape}")
print(f"test shape: {test.shape}")

train shape: (122375, 40)
val shape: (8784, 40)
test shape: (13897, 40)


In [13]:
print(model_features(pjme).items())

dict_items([('sarimax', ['is_weekend', 'is_holiday', 'mean_cdh', 'mean_hdh']), ('dhr', ['sin_24_k3', 'cos_24_k3', 'sin_168_k3', 'cos_168_k3', 'sin_8766_k2', 'cos_8766_k2', 'is_weekend', 'is_holiday', 'mean_cdh', 'mean_hdh']), ('ml', ['cdh_PHL', 'hdh_PHL', 'cdh_EWR', 'hdh_EWR', 'cdh_IAD', 'hdh_IAD', 'cdh_BWI', 'hdh_BWI', 'cdh_ILM', 'hdh_ILM', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'lag_24', 'lag_168', 'roll_mean_24', 'roll_mean_168', 'is_weekend', 'is_holiday'])])
